This is a comparison between two container implementations: container-as-table and container-as-base-value.

In [1]:
import random

NUM_WVS = 100000
NUM_LEADING_ZEROS_WVS = 50000

random.seed(42)
# generate random wire vectors, each containing 1 to 32 random integers
wvs = [[random.randint(0, 5000) for _ in range(random.randint(1, 32))] for _ in range(NUM_WVS)]
# and random wire vectors with leading zeros
wvs += [[random.randint(0, 5000) for _ in range(random.randint(1, 16))] + [0 for _ in range(random.randint(1, 16))] for _ in range(NUM_LEADING_ZEROS_WVS)]

In [2]:
# container-as-table: normalized representation of variable-length lists of integers
import sqlite3
import time

db = sqlite3.connect(":memory:")

db.execute("""
    CREATE TABLE IF NOT EXISTS wirevecs (
        id INTEGER PRIMARY KEY,
        hash INTEGER NOT NULL,
        length INTEGER NOT NULL
    );
""")
db.execute("""
    CREATE TABLE IF NOT EXISTS wirevec_members (
        wirevec_id INTEGER,
        wire_id INTEGER,
        idx INTEGER,
        PRIMARY KEY (wirevec_id, idx)
    );
""")

db.execute("CREATE INDEX IF NOT EXISTS wirevecs_hash_idx ON wirevecs (hash);")
db.execute("CREATE INDEX IF NOT EXISTS wirevec_members_wire_id_idx ON wirevec_members (wire_id);")
db.execute("CREATE INDEX IF NOT EXISTS wirevec_members_wirevec_id_idx ON wirevec_members (wirevec_id);")

def get_wirevec(db: sqlite3.Connection, wirevec_id: int) -> list[int] | None:
    cur = db.execute("SELECT 1 FROM wirevecs WHERE id = ?", (wirevec_id,))
    if cur.fetchone() is None:
        return None
    cur = db.execute("SELECT wire_id FROM wirevec_members WHERE wirevec_id = ? ORDER BY idx", (wirevec_id,))
    return [row[0] for row in cur]

def add_wirevec(db: sqlite3.Connection, wires: list[int]) -> int:
    hash_ = hash(tuple(wires))
    cur = db.execute("SELECT id FROM wirevecs WHERE hash = ?", (hash_,))
    for (wirevec_id,) in cur:
        if get_wirevec(db, wirevec_id) == wires:
            return wirevec_id
    cur = db.execute("INSERT INTO wirevecs (hash, length) VALUES (?, ?)", (hash_, len(wires)))
    wirevec_id = cur.lastrowid
    for idx, wire_id in enumerate(wires):
        db.execute("INSERT INTO wirevec_members (wirevec_id, wire_id, idx) VALUES (?, ?, ?)", (wirevec_id, wire_id, idx))
    return wirevec_id

start = time.time()
for wv in wvs:
    add_wirevec(db, wv)
print(f"Build time: {time.time() - start:.2f} seconds")

start = time.time()
cnt = 0
cur = db.execute("""
    SELECT wirevec_id FROM wirevecs JOIN wirevec_members
    ON wirevecs.id = wirevec_members.wirevec_id
    WHERE wirevec_members.idx = wirevecs.length - 1 AND wirevec_members.wire_id = 0
""")
for _ in cur:
    cnt += 1
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} wirevecs with leading 0")

start = time.time()
cnt = 0
cur = db.execute("""
    SELECT wirevecs.id FROM wirevecs JOIN wirevec_members AS member1 JOIN wirevec_members member2
    ON wirevecs.id = member1.wirevec_id AND wirevecs.id = member2.wirevec_id
    WHERE member1.idx = wirevecs.length - 1 AND member2.idx = wirevecs.length - 2 AND member1.wire_id = member2.wire_id
""")

for _ in cur:
    cnt += 1
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} wirevecs with last two wires equal")

start = time.time()
cnt = 0
cur = db.execute("""
    SELECT wv1.id, wv2.id FROM wirevecs AS wv1 JOIN wirevecs AS wv2 JOIN wirevec_members AS w1 JOIN wirevec_members AS w2
    ON wv1.id = w1.wirevec_id AND wv2.id = w2.wirevec_id AND w1.wire_id = w2.wire_id
    WHERE wv1.length = wv2.length AND w1.idx = 0 AND w2.idx = 0 AND wv1.id < wv2.id
""")
for _ in cur:
    cnt += 1
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} pairs of wirevecs with same length and first wire equal")

start = time.time()
for _ in range(100):
    wire_id = random.randint(1, 5000)
    cur = db.execute("SELECT wirevec_id FROM wirevec_members WHERE wire_id = ?", (wire_id,))
    for wv_id in cur.fetchall():
        db.execute("UPDATE wirevecs SET hash = hash + 1 WHERE id = ?", (wv_id[0],)) # simulate hash update
        db.execute("UPDATE wirevec_members SET wire_id = 0 WHERE wirevec_id = ? AND wire_id = ?", (wv_id[0], wire_id))
print(f"Update time: {time.time() - start:.2f} seconds")

Build time: 26.39 seconds
Query time: 0.20 seconds, found 49967 wirevecs with leading 0
Query time: 0.74 seconds, found 46800 wirevecs with last two wires equal
Query time: 79.11 seconds, found 72232 pairs of wirevecs with same length and first wire equal
Update time: 1.05 seconds


In [3]:
# container-as-base-value
import sqlite3
import time

db = sqlite3.connect(":memory:")

db.execute("""
    CREATE TABLE IF NOT EXISTS wirevecs (
        id INTEGER PRIMARY KEY,
        hash INTEGER NOT NULL,
        members VARCHAR(256) NOT NULL
    );
""")

db.execute("CREATE INDEX IF NOT EXISTS wirevecs_hash_idx ON wirevecs (hash);")

def get_wirevec(db: sqlite3.Connection, wirevec_id: int) -> list[int] | None:
    cur = db.execute("SELECT members FROM wirevecs WHERE id = ?", (wirevec_id,))
    row = cur.fetchone()
    if row is None:
        return None
    return list(map(int, row[0].split(",")))

def add_wirevec(db: sqlite3.Connection, wires: list[int]) -> int:
    hash_ = hash(tuple(wires))
    cur = db.execute("SELECT id FROM wirevecs WHERE hash = ?", (hash_,))
    for (wirevec_id,) in cur:
        if get_wirevec(db, wirevec_id) == wires:
            return wirevec_id
    cur = db.execute("INSERT INTO wirevecs (hash, members) VALUES (?, ?)", (hash_, ",".join(map(str, wires))))
    return cur.lastrowid

start = time.time()
for wv in wvs:
    add_wirevec(db, wv)
print(f"Build time: {time.time() - start:.2f} seconds")

start = time.time()
cnt = 0
cur = db.execute("SELECT id FROM wirevecs WHERE members LIKE '%,0'")
for _ in cur:
    cnt += 1
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} wirevecs with leading 0")

start = time.time()
cnt = 0
cur = db.execute("SELECT id, members FROM wirevecs")
for (wirevec_id, members) in cur:
    wires = list(map(int, members.split(",")))
    if len(wires) >= 2 and wires[-1] == wires[-2]:
        cnt += 1
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} wirevecs with last two wires equal")

start = time.time()
cnt = 0
cur = db.execute("SELECT members FROM wirevecs ORDER BY id")
wirevecs = []
for (members,) in cur:
    wires = list(map(int, members.split(",")))
    wirevecs.append(wires)
firsts: dict[tuple[int, int], list[list[int]]] = {}
for wirevec in wirevecs:
    if len(wirevec) == 0:
        continue
    first = wirevec[0]
    length = len(wirevec)
    if (first, length) in firsts:
        firsts[(first, length)].append(wirevec)
    else:
        firsts[(first, length)] = [wirevec]
for k, v in firsts.items():
    n = len(v)
    if n >= 2:
        cnt += n * (n - 1) // 2
print(f"Query time: {time.time() - start:.2f} seconds, found {cnt} pairs of wirevecs with same length and first wire equal")

start = time.time()
for _ in range(100):
    wire_id = random.randint(1, 5000)
    cur = db.execute("SELECT id, members FROM wirevecs")
    for (wv_id, members) in cur:
        wires = list(map(int, members.split(",")))
        if wire_id in wires:
            wires = [0 if w == wire_id else w for w in wires]
            db.execute("UPDATE wirevecs SET hash = hash + 1, members = ? WHERE id = ?", (",".join(map(str, wires)), wv_id)) # simulate hash update
print(f"Update time: {time.time() - start:.2f} seconds")

Build time: 3.08 seconds
Query time: 0.13 seconds, found 49966 wirevecs with leading 0
Query time: 0.87 seconds, found 46800 wirevecs with last two wires equal
Query time: 1.37 seconds, found 72232 pairs of wirevecs with same length and first wire equal
Update time: 86.17 seconds
